# Chile (CLP) — Fixed Income Derivatives

Cámara swap (TPM-based), BCP nominal bonds, BCU (UF-linked inflation bonds), and breakeven inflation.

**Key features:**
- TPM: Tasa de Política Monetaria (BCCh overnight rate)
- UF: Unidad de Fomento — daily inflation unit (~CLP 37,000)
- BCU bonds denominated in UF (real terms)
- Dual-curve pricing: CLP nominal + UF real

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.fixed_income.chilean import (
    CamaraSwap, BCPBond, BCUBond,
    build_clp_curve, build_uf_curve, breakeven_inflation,
    synthetic_clp_strip, synthetic_uf_strip,
)
from pricebook.fixed_income.inflation_unit import get_inflation_unit, dual_curve_breakeven
from pricebook.viz import configure_theme, greeks_profile
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

REF = date(2024, 11, 4)
print(f"Reference date: {REF}")
print(f"TPM (BCCh policy rate): 5.50%")
print(f"UF value: ~CLP 37,900")

## 1. CLP Nominal & UF Real Curves

In [ ]:
# Build dual curves
clp_strip = synthetic_clp_strip(REF, tpm=0.0550)
uf_strip = synthetic_uf_strip(REF, real_rate=0.02)

clp_curve = build_clp_curve(REF, clp_strip)
uf_curve = build_uf_curve(REF, uf_strip)

print(f"CLP Nominal Curve ({len(clp_strip)} points):")
print(f"{'Tenor':>8}  {'Rate':>8}  {'DF':>10}")
print(f"{'─'*8}  {'─'*8}  {'─'*10}")
for c in clp_strip:
    print(f"{c['months']:>6}M  {c['rate']*100:>7.2f}%  {clp_curve.df(c['maturity']):>10.6f}")

print(f"\nUF Real Curve ({len(uf_strip)} points):")
print(f"{'Tenor':>8}  {'Rate':>8}  {'DF':>10}")
print(f"{'─'*8}  {'─'*8}  {'─'*10}")
for c in uf_strip:
    print(f"{c['months']:>6}M  {c['rate']*100:>7.2f}%  {uf_curve.df(c['maturity']):>10.6f}")

In [ ]:
# Dual curve plot
with apply_theme():
    fig, (ax1, ax2) = create_figure(2)

    clp_y = [c["years"] for c in clp_strip]
    uf_y = [c["years"] for c in uf_strip]

    ax1.plot(clp_y, [c["rate"]*100 for c in clp_strip], 'o-', label='CLP Nominal', linewidth=2)
    ax1.plot(uf_y, [c["rate"]*100 for c in uf_strip], 's-', label='UF Real', linewidth=2)
    ax1.set_xlabel("Tenor (years)")
    ax1.set_ylabel("Rate (%)")
    ax1.set_title("Chile — Nominal vs Real Yield Curves")
    ax1.legend()

    ax2.plot(clp_y, [clp_curve.df(c["maturity"]) for c in clp_strip], 'o-', label='CLP', linewidth=2)
    ax2.plot(uf_y, [uf_curve.df(c["maturity"]) for c in uf_strip], 's-', label='UF', linewidth=2)
    ax2.set_xlabel("Tenor (years)")
    ax2.set_ylabel("Discount Factor")
    ax2.set_title("Discount Factors")
    ax2.legend()

    fig.tight_layout()

## 2. Cámara Swap (TPM Overnight)

In [ ]:
# Cámara swap pricing
print(f"Cámara Swap Pricing (CLP 10bn notional):")
print(f"{'Tenor':>6}  {'Par Rate':>10}  {'DV01 (CLP)':>12}")
print(f"{'─'*6}  {'─'*10}  {'─'*12}")

for t in [1, 2, 3, 5, 7, 10]:
    swap = CamaraSwap(REF, REF + relativedelta(years=t), 0.0550, notional=10e9)
    r = swap.price(clp_curve)
    print(f"{t:>4}Y  {r.par_rate*100:>9.2f}%  {r.dv01:>12,.0f}")

## 3. BCP Nominal Bond & BCU (UF-Linked)

In [ ]:
# BCP (nominal CLP) and BCU (UF-linked) comparison
current_uf = 37_900.0
print(f"Bond Pricing (BCP nominal vs BCU real, UF = {current_uf:,.0f} CLP):")
print(f"{'Type':>6}  {'Tenor':>6}  {'Coupon':>8}  {'Price':>10}  {'Notes':>20}")
print(f"{'─'*6}  {'─'*6}  {'─'*8}  {'─'*10}  {'─'*20}")

for t in [3, 5, 10, 20]:
    bcp = BCPBond(REF, REF + relativedelta(years=t), 0.055)
    bcp_px = bcp.dirty_price(clp_curve)
    print(f"{'BCP':>6}  {t:>4}Y  {5.50:>7.2f}%  {bcp_px:>10.4f}  CLP per 100")

    bcu = BCUBond(REF, REF + relativedelta(years=t), 0.025, face_uf=1000)
    r = bcu.price(REF, uf_curve, current_uf)
    print(f"{'BCU':>6}  {t:>4}Y  {2.50:>7.2f}%  {r.real_price:>10.2f}  UF ({r.nominal_price/1e6:>.1f}M CLP)")

## 4. Breakeven Inflation

In [ ]:
# BEI term structure
bei = breakeven_inflation(clp_curve, uf_curve)
print(f"Chile Breakeven Inflation (CLP nominal vs UF real):")
print(f"{'Tenor':>6}  {'Nominal':>8}  {'Real':>8}  {'BEI':>8}")
print(f"{'─'*6}  {'─'*8}  {'─'*8}  {'─'*8}")
for row in bei:
    print(f"{row['maturity_years']:>4}Y  {row['nominal_rate']*100:>7.2f}%  "
          f"{row['real_rate']*100:>7.2f}%  {row['breakeven_inflation']*100:>7.2f}%")

with apply_theme():
    fig, axes = create_figure(1)
    ax = axes[0]
    years = [r["maturity_years"] for r in bei]
    ax.plot(years, [r["nominal_rate"]*100 for r in bei], 'o-', label='Nominal (CLP)', linewidth=2)
    ax.plot(years, [r["real_rate"]*100 for r in bei], 's-', label='Real (UF)', linewidth=2)
    ax.fill_between(years, [r["real_rate"]*100 for r in bei],
                     [r["nominal_rate"]*100 for r in bei], alpha=0.2, label='BEI')
    ax.set_xlabel("Tenor (years)")
    ax.set_ylabel("Rate (%)")
    ax.set_title("Chile — Breakeven Inflation Term Structure")
    ax.legend()
    fig.tight_layout()

## Summary

| Instrument | Convention | Key Feature |
|---|---|---|
| Cámara Swap | TPM-based, ACT/360 | Overnight rate swap |
| BCP | Semi-annual, ACT/365 | Nominal CLP sovereign |
| BCU | Semi-annual, UF-denominated | Real sovereign (UF) |
| BEI | CLP curve - UF curve | ~3.5% implied inflation |